# ============================================================
# EDA — Primary Dataset
# ============================================================
 Goal:
Analyse the Kaggle/Datafiniti primary dataset before merging it
with the larger Amazon Reviews 2023 dataset.

This helps us understand:
- class imbalance
- product concentration
- category concentration
- review length patterns
- whether the primary dataset is enough on its own
============================================================

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ------------------------------------------------------------
# 1. Create EDA copy
# ------------------------------------------------------------

primary_eda = primary_df.copy()

# Ensure rating is numeric
primary_eda["rating"] = pd.to_numeric(primary_eda["rating"], errors="coerce")

# Create sentiment label using existing helper function
primary_eda["sentiment"] = primary_eda["rating"].apply(map_rating_to_sentiment)

# Create review length features
primary_eda["review_word_count"] = primary_eda["review_text"].fillna("").str.split().apply(len)
primary_eda["review_char_count"] = primary_eda["review_text"].fillna("").str.len()

# Useful order for plots
sentiment_order = ["negative", "neutral", "positive"]

print("PRIMARY DATASET — BASIC OVERVIEW")
print("=" * 50)
print("Shape:", primary_eda.shape)
display(primary_eda.head())

In [ ]:
# ------------------------------------------------------------
# 2. Missing values
# ------------------------------------------------------------

print("\nMISSING VALUES")
print("=" * 50)

eda_columns = [
    "review_text",
    "rating",
    "product_name",
    "raw_category",
    "source",
    "sentiment"
]

missing_values = primary_eda[eda_columns].isna().sum().to_frame("missing_count")
missing_values["percentage"] = (missing_values["missing_count"] / len(primary_eda) * 100).round(2)

display(missing_values.sort_values("percentage", ascending=False))

plt.figure(figsize=(9, 4))
missing_values["percentage"].sort_values(ascending=False).plot(kind="bar")
plt.title("Primary Dataset — Missing Values by Column (%)")
plt.ylabel("Missing values (%)")
plt.xlabel("Column")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
# ------------------------------------------------------------
# 3. Duplicate reviews
# ------------------------------------------------------------

print("\nDUPLICATE REVIEWS")
print("=" * 50)

duplicate_count = primary_eda.duplicated(
    subset=["review_text", "rating", "product_name"]
).sum()

duplicate_pct = duplicate_count / len(primary_eda) * 100

print(f"Duplicate rows: {duplicate_count}")
print(f"Duplicate percentage: {duplicate_pct:.2f}%")

In [ ]:
# ------------------------------------------------------------
# 4. Rating distribution
# ------------------------------------------------------------

print("\nRATING DISTRIBUTION")
print("=" * 50)

rating_distribution = (
    primary_eda["rating"]
    .value_counts()
    .sort_index()
    .to_frame("count")
)

rating_distribution["percentage"] = (
    rating_distribution["count"] / len(primary_eda) * 100
).round(2)

display(rating_distribution)

plt.figure(figsize=(7, 4))
sns.countplot(
    data=primary_eda,
    x="rating",
    order=sorted(primary_eda["rating"].dropna().unique())
)
plt.title("Primary Dataset — Star Rating Distribution")
plt.xlabel("Star rating")
plt.ylabel("Number of reviews")
plt.show()

In [ ]:
# ------------------------------------------------------------
# 5. Sentiment distribution
# ------------------------------------------------------------

print("\nSENTIMENT DISTRIBUTION")
print("=" * 50)

sentiment_distribution = (
    primary_eda["sentiment"]
    .value_counts()
    .reindex(sentiment_order)
    .fillna(0)
    .astype(int)
    .to_frame("count")
)

sentiment_distribution["percentage"] = (
    sentiment_distribution["count"] / len(primary_eda) * 100
).round(2)

display(sentiment_distribution)

plt.figure(figsize=(7, 4))
sns.countplot(
    data=primary_eda,
    x="sentiment",
    order=sentiment_order
)
plt.title("Primary Dataset — Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of reviews")
plt.show()

In [ ]:
# ------------------------------------------------------------
# 6. Product concentration
# ------------------------------------------------------------

print("\nTOP 20 PRODUCTS BY REVIEW COUNT")
print("=" * 50)

top_products = (
    primary_eda["product_name"]
    .value_counts()
    .head(20)
    .to_frame("review_count")
)

top_products["percentage"] = (
    top_products["review_count"] / len(primary_eda) * 100
).round(2)

display(top_products)

plt.figure(figsize=(10, 7))
top_products["review_count"].sort_values().plot(kind="barh")
plt.title("Primary Dataset — Top 20 Products by Review Count")
plt.xlabel("Number of reviews")
plt.ylabel("Product")
plt.show()

In [ ]:
# ------------------------------------------------------------
# 7. Category concentration
# ------------------------------------------------------------

print("\nTOP 20 CATEGORIES BY REVIEW COUNT")
print("=" * 50)

top_categories = (
    primary_eda["raw_category"]
    .value_counts()
    .head(20)
    .to_frame("review_count")
)

top_categories["percentage"] = (
    top_categories["review_count"] / len(primary_eda) * 100
).round(2)

display(top_categories)

plt.figure(figsize=(10, 7))
top_categories["review_count"].sort_values().plot(kind="barh")
plt.title("Primary Dataset — Top 20 Categories by Review Count")
plt.xlabel("Number of reviews")
plt.ylabel("Category")
plt.show()

In [ ]:
# ------------------------------------------------------------
# 8. Sentiment by top products
# ------------------------------------------------------------

print("\nSENTIMENT DISTRIBUTION FOR TOP 10 PRODUCTS (%)")
print("=" * 50)

top_product_names = primary_eda["product_name"].value_counts().head(10).index

product_sentiment = pd.crosstab(
    primary_eda[primary_eda["product_name"].isin(top_product_names)]["product_name"],
    primary_eda[primary_eda["product_name"].isin(top_product_names)]["sentiment"],
    normalize="index"
).reindex(columns=sentiment_order).fillna(0)

display((product_sentiment * 100).round(2))

(product_sentiment * 100).plot(
    kind="barh",
    stacked=True,
    figsize=(10, 6)
)
plt.title("Primary Dataset — Sentiment Distribution for Top 10 Products (%)")
plt.xlabel("Percentage")
plt.ylabel("Product")
plt.legend(title="Sentiment")
plt.show()

In [ ]:
# ------------------------------------------------------------
# 9. Sentiment by top categories
# ------------------------------------------------------------

print("\nSENTIMENT DISTRIBUTION FOR TOP 10 CATEGORIES (%)")
print("=" * 50)

top_category_names = primary_eda["raw_category"].value_counts().head(10).index

category_sentiment = pd.crosstab(
    primary_eda[primary_eda["raw_category"].isin(top_category_names)]["raw_category"],
    primary_eda[primary_eda["raw_category"].isin(top_category_names)]["sentiment"],
    normalize="index"
).reindex(columns=sentiment_order).fillna(0)

display((category_sentiment * 100).round(2))

(category_sentiment * 100).plot(
    kind="barh",
    stacked=True,
    figsize=(10, 6)
)
plt.title("Primary Dataset — Sentiment Distribution for Top 10 Categories (%)")
plt.xlabel("Percentage")
plt.ylabel("Category")
plt.legend(title="Sentiment")
plt.show()

In [ ]:
# ------------------------------------------------------------
# 10. Review length analysis
# ------------------------------------------------------------

print("\nREVIEW LENGTH STATISTICS BY SENTIMENT")
print("=" * 50)

review_length_stats = (
    primary_eda
    .groupby("sentiment")[["review_word_count", "review_char_count"]]
    .describe()
    .round(2)
)

display(review_length_stats)

plt.figure(figsize=(8, 4))
sns.boxplot(
    data=primary_eda,
    x="sentiment",
    y="review_word_count",
    order=sentiment_order
)
plt.title("Primary Dataset — Review Word Count by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Word count")
plt.ylim(0, primary_eda["review_word_count"].quantile(0.95))
plt.show()

In [ ]:
# ------------------------------------------------------------
# 11. Rating by review length
# ------------------------------------------------------------

plt.figure(figsize=(8, 4))
sns.boxplot(
    data=primary_eda,
    x="rating",
    y="review_word_count",
    order=sorted(primary_eda["rating"].dropna().unique())
)
plt.title("Primary Dataset — Review Word Count by Rating")
plt.xlabel("Rating")
plt.ylabel("Word count")
plt.ylim(0, primary_eda["review_word_count"].quantile(0.95))
plt.show()

In [ ]:
# ------------------------------------------------------------
# 12. Average rating by category
# ------------------------------------------------------------

print("\nAVERAGE RATING BY CATEGORY")
print("=" * 50)

category_rating_stats = (
    primary_eda
    .groupby("raw_category")
    .agg(
        avg_rating=("rating", "mean"),
        review_count=("review_text", "count")
    )
    .query("review_count >= 10")
    .sort_values("avg_rating", ascending=False)
)

print("Highest-rated categories with at least 10 reviews:")
display(category_rating_stats.head(10).round(2))

print("Lowest-rated categories with at least 10 reviews:")
display(category_rating_stats.tail(10).round(2))

In [ ]:
# ------------------------------------------------------------
# 13. Final EDA summary
# ------------------------------------------------------------

print("\nPRIMARY DATASET — EDA SUMMARY")
print("=" * 50)

majority_class = sentiment_distribution["count"].idxmax()
majority_pct = sentiment_distribution.loc[majority_class, "percentage"]

top_product = primary_eda["product_name"].value_counts().idxmax()
top_product_count = primary_eda["product_name"].value_counts().max()
top_product_pct = top_product_count / len(primary_eda) * 100

top_category = primary_eda["raw_category"].value_counts().idxmax()
top_category_count = primary_eda["raw_category"].value_counts().max()
top_category_pct = top_category_count / len(primary_eda) * 100

missing_review_pct = primary_eda["review_text"].isna().mean() * 100
missing_rating_pct = primary_eda["rating"].isna().mean() * 100

print(f"Dataset size: {primary_eda.shape[0]} rows and {primary_eda.shape[1]} columns.")
print(f"Missing review text: {missing_review_pct:.2f}%.")
print(f"Missing rating: {missing_rating_pct:.2f}%.")
print(f"Majority sentiment class: {majority_class} ({majority_pct:.2f}%).")
print(f"Most reviewed product: {top_product} ({top_product_count} reviews, {top_product_pct:.2f}%).")
print(f"Most common category: {top_category} ({top_category_count} reviews, {top_category_pct:.2f}%).")